# A small PVC model for the device — chosen on DS1, tested on DS2 (Notebook 13)
**Bektur Asilov · COMSE-23 · Ala-Too**

Notebook 11 showed ventricular ectopic beats (PVC) can be found from beat shape + timing (94.7 % / 91.8 % on DS2)
with a 300-tree model. The device needs something small, and the server and the device must compute **exactly the
same features**. So the features are now the server module `backend/app/ectopy.py` (pasted unchanged below) on the
beats of the server/device R-peak detector `ecgtools.r_peaks`.

Rules fixed **before** the run:
* task: ventricular ectopic beat (V, E) vs any other beat (N, L, R, e, j, A, a, J, S); F and Q beats and beats that
  match no label within 150 ms are left out; paced records excluded; split DS1 / DS2 (de Chazal 2004);
* three candidates: (A) 50 trees of depth ≤ 3, (B) 100 trees of depth ≤ 4 (both `HistGradientBoostingClassifier`,
  `class_weight='balanced'`, `random_state=42`), (C) logistic regression on standardised features (`class_weight='balanced'`);
* the candidate with the best V F1 in **5-fold cross-validation over DS1 records** (GroupKFold) is chosen —
  DS2 is not used for the choice; threshold 0.5;
* reported on DS2 for all three (the chosen one is the result): V sensitivity, PPV, per-record PVC burden;
* the chosen model is exported to plain numbers and the exported version is checked to give the same probabilities.

> Kaggle: Internet **On**, CPU.

In [ ]:
import os, time, json, subprocess, sys
import numpy as np, pandas as pd
from scipy.signal import butter, sosfilt, resample_poly
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb'], check=True)
import wfdb
FS = 125
OUT_DIR = os.environ.get('OUT_DIR', '/kaggle/working')
DB = os.environ.get('MITDB_DIR')
RECS = [100,101,102,103,104,105,106,107,108,109,111,112,113,114,115,116,117,118,119,121,122,123,124,
        200,201,202,203,205,207,208,209,210,212,213,214,215,217,219,220,221,222,223,228,230,231,232,233,234]
if not DB:
    DB = '/kaggle/tmp/mitdb'; os.makedirs(DB, exist_ok=True)
    for r in RECS:
        for ext in ('hea', 'dat', 'atr'):
            for base in ('https://physionet-open.s3.amazonaws.com/mitdb/1.0.0/', 'https://physionet.org/files/mitdb/1.0.0/'):
                if subprocess.run(['curl', '-fsSL', '--max-time', '300', '-o', f'{DB}/{r}.{ext}', base + f'{r}.{ext}']).returncode == 0: break
            else:
                raise RuntimeError(f'download failed: {r}.{ext}')
print('records:', len(RECS))

## Server code, unchanged: `ecgtools.py` (detector) and `ectopy.py` (features, model evaluation)

In [ ]:
"""ECG strip analysis on the server: R-peaks, RR intervals, Poincaré points, signal quality index, PNG image.

Pure Python (strips are short: 10 s x 125 Hz). The R-peak detector is the firmware algorithm
(derivative^2 -> 150 ms moving average -> adaptive threshold, 250 ms refractory), adapted to run offline
on a short strip: the threshold tracker is primed with the first 2 s instead of a 2-s warm-up.
"""

import io
import math
import statistics


def _highpass(x: list[float], fs: int) -> list[float]:
    """Remove baseline wander: subtract a 0.6-s centred moving mean."""
    w = max(3, int(0.6 * fs)) | 1
    h = w // 2
    n = len(x)
    pre = [0.0]
    for v in x:
        pre.append(pre[-1] + v)
    out = []
    for i in range(n):
        a, b = max(0, i - h), min(n, i + h + 1)
        out.append(x[i] - (pre[b] - pre[a]) / (b - a))
    return out


def r_peaks(x: list[float], fs: int = 125) -> list[int]:
    n = len(x)
    if n < fs:
        return []
    y = _highpass(x, fs)
    ma = max(3, int(0.15 * fs))
    env, buf, s, prev = [], [0.0] * ma, 0.0, y[0]
    for i, v in enumerate(y):
        e = (v - prev) ** 2
        prev = v
        s += e - buf[i % ma]
        buf[i % ma] = e
        env.append(s / ma)
    decay = 0.5 ** (1 / (2.0 * fs))
    thr, refr = 0.35, int(0.25 * fs)
    pk = max(env[: min(n, 2 * fs)]) or 1e-12
    out, last, armed = [], -10 ** 9, True
    for i in range(int(0.2 * fs), n):
        pk = max(pk * decay, env[i])
        if armed and env[i] > thr * pk and i - last > refr:
            # the envelope lags the QRS by ~MA/2: take the largest |amplitude| just before
            a, b = max(0, i - ma - 4), min(n, i + 3)
            j = max(range(a, b), key=lambda k: abs(y[k]))
            if not out or j - out[-1] > refr:
                out.append(j)
            last, armed = i, False
        elif not armed and env[i] < 0.5 * thr * pk:
            armed = True
    return out


def rr_intervals(peaks: list[int], fs: int = 125) -> list[float]:
    return [round((b - a) / fs, 3) for a, b in zip(peaks, peaks[1:])]


def poincare(rr: list[float]) -> dict:
    """Points (RRn, RRn+1) and the SD1/SD2 descriptors (SD1 = short-term, beat-to-beat variability)."""
    rr = [r for r in rr if 0.25 <= r <= 2.0]
    return poincare_points([[a, b] for a, b in zip(rr, rr[1:])])


def poincare_points(pts: list[list[float]]) -> dict:
    if len(pts) < 3:
        return {"points": pts, "sd1": None, "sd2": None, "ratio": None}
    d1 = [(b - a) / math.sqrt(2) for a, b in pts]
    d2 = [(b + a) / math.sqrt(2) for a, b in pts]
    sd1, sd2 = statistics.pstdev(d1), statistics.pstdev(d2)
    return {"points": pts, "sd1": round(sd1 * 1000, 1), "sd2": round(sd2 * 1000, 1),
            "ratio": round(sd1 / sd2, 2) if sd2 > 0 else None}


def irregularity(rr: list[float]) -> float | None:
    """Mean |successive difference| / mean RR — high (>~0.1) in AF, low in sinus rhythm."""
    rr = [r for r in rr if 0.25 <= r <= 2.0]
    if len(rr) < 4:
        return None
    return round(statistics.mean(abs(b - a) for a, b in zip(rr, rr[1:])) / statistics.mean(rr), 3)


def quality(x: list[float], fs: int = 125, peaks: list[int] | None = None) -> dict:
    """Signal quality index 0..1 from four checks a cardiologist does by eye:
    no flat line / clipping, plausible heart rate, consistent QRS shape, little high-frequency noise."""
    n = len(x)
    if n < 2 * fs:
        return {"sqi": 0.0, "level": "poor", "reasons": ["too_short"]}
    peaks = r_peaks(x, fs) if peaks is None else peaks
    y = _highpass(x, fs)
    reasons = []
    rng = max(y) - min(y)
    flat = sum(1 for a, b in zip(y, y[1:]) if abs(b - a) < 1e-4) / n
    s_flat = 0.0 if rng < 0.05 or flat > 0.5 else 1.0
    if s_flat == 0:
        reasons.append("flat")
    secs = n / fs
    bpm = len(peaks) / secs * 60
    s_rate = 1.0 if 30 <= bpm <= 200 else 0.0
    if not s_rate:
        reasons.append("rate")
    # QRS template correlation
    h = int(0.1 * fs)
    segs = [y[p - h:p + h] for p in peaks if p - h >= 0 and p + h <= n]
    if len(segs) >= 3:
        tmpl = [statistics.median(col) for col in zip(*segs)]

        def corr(a, b):
            ma, mb = statistics.mean(a), statistics.mean(b)
            num = sum((u - ma) * (v - mb) for u, v in zip(a, b))
            den = math.sqrt(sum((u - ma) ** 2 for u in a) * sum((v - mb) ** 2 for v in b)) or 1e-12
            return num / den
        s_shape = max(0.0, statistics.mean(corr(s, tmpl) for s in segs))
    else:
        s_shape = 0.0
    if s_shape < 0.7:
        reasons.append("shape")
    # high-frequency noise: energy of the second difference relative to the signal
    d2 = [y[i + 1] - 2 * y[i] + y[i - 1] for i in range(1, n - 1)]
    noise = (sum(v * v for v in d2) / len(d2)) ** 0.5 / (statistics.pstdev(y) or 1e-12)
    s_noise = max(0.0, min(1.0, 1.5 - noise))
    if s_noise < 0.5:
        reasons.append("noise")
    sqi = s_flat * s_rate * (0.6 * s_shape + 0.4 * s_noise)
    sqi = round(max(0.0, min(1.0, sqi)), 2)
    return {"sqi": sqi, "level": "good" if sqi >= 0.7 else "fair" if sqi >= 0.4 else "poor", "reasons": reasons}


PAUSE_S = 3.0   # an RR gap of 3 s or more is a pause worth showing to a doctor


def pauses(peaks: list[int], fs: int = 125) -> list[dict]:
    return [{"at_s": round(a / fs, 2), "rr_s": round((b - a) / fs, 2)}
            for a, b in zip(peaks, peaks[1:]) if (b - a) / fs >= PAUSE_S]


def analyze(x: list[float], fs: int = 125) -> dict:
    p = r_peaks(x, fs)
    rr = rr_intervals(p, fs)
    q = quality(x, fs, p)
    # a pause only counts when the rest of the strip is clean; otherwise it is more likely a lost electrode
    return {"r_peaks": p, "rr": rr, "poincare": poincare(rr), "irregularity": irregularity(rr),
            "quality": q, "pauses": pauses(p, fs) if q["sqi"] >= 0.4 else []}


def _movmean(x: list[float], w: int) -> list[float]:
    """Centred moving mean over w samples (w odd), shrinking at the edges — the same as in _highpass."""
    h, n = w // 2, len(x)
    pre = [0.0]
    for v in x:
        pre.append(pre[-1] + v)
    return [(pre[min(n, i + h + 1)] - pre[max(0, i - h)]) / (min(n, i + h + 1) - max(0, i - h)) for i in range(n)]


# second detector settings (smooth samples, baseline window s, threshold fraction, refractory s)
AMP_DEFAULT = (3, 0.12, 0.5, 0.30)   # v3: best F1 on MIT-BIH of 16 settings (Notebook 9); v2 was (1, 0.6, 0.5, 0.25)


def r_peaks_amp(x: list[float], fs: int = 125, params: tuple = AMP_DEFAULT) -> list[int]:
    """Second, independent R-peak detector for the agreement check: amplitude of the signal after removing
    the baseline (not the slope energy the main detector uses), adaptive threshold at a fraction of the
    decaying peak, refractory period. Optional light smoothing and a short baseline window make it a crude
    QRS band-pass that ignores T-waves better."""
    w1, w2s, frac, refr_s = params
    n = len(x)
    if n < fs:
        return []
    y = _movmean(x, w1) if w1 > 1 else list(x)
    w2 = max(3, int(w2s * fs)) | 1
    base = _movmean(y, w2)
    a = [abs(u - b) for u, b in zip(y, base)]
    decay = 0.5 ** (1 / (2.0 * fs))
    refr, look = int(refr_s * fs), int(0.1 * fs)
    pk = max(a[: min(n, 2 * fs)]) or 1e-12
    out, i = [], 0
    while i < n:
        pk = max(pk * decay, a[i])
        if a[i] > frac * pk and (not out or i - out[-1] > refr):
            j = max(range(i, min(n, i + look)), key=lambda k: a[k])
            if not out or j - out[-1] > refr:
                out.append(j)
            i = j + 1
            continue
        i += 1
    return out


def bsqi(p1: list[int], p2: list[int], fs: int = 125, tol_s: float = 0.15) -> float:
    """Agreement of two detectors (Li, Mark & Clifford 2008): matched / (n1 + n2 - matched), match within 150 ms."""
    if not p1 and not p2:
        return 0.0
    tol, used, m = int(tol_s * fs), set(), 0
    j = 0
    for p in p1:
        while j < len(p2) and p2[j] < p - tol:
            j += 1
        k = j
        while k < len(p2) and p2[k] <= p + tol:
            if k not in used:
                used.add(k)
                m += 1
                break
            k += 1
    return round(m / (len(p1) + len(p2) - m), 3)


# "Check me in 30 seconds" (wellness wording), see ml/CHECK30_CINC2017.md and ml/CHECK30_BSQI.md
# v1 (CinC 2017, never tuned on it): 91.4 % of AF recordings irregular, 14.1 % of normal ones too; noise not caught.
# v2 adds the two-detector agreement gate (bSQI >= 0.8), rules fixed before Notebook 8: false "irregular" on clean
# normal rhythm 14.1 % -> 3.3 %, noise caught (97.5 % of noisy recordings -> "could not read"), but 44 % of clean
# normal checks need a repeat and AF found drops 91.4 -> 69.4 %. See ml/CHECK30_BSQI.md.
# v3 (current): second detector chosen on MIT-BIH (Notebook 9). CinC 2017 clean: AF found 85.6 %, normal called
# irregular 8.9 %, repeats 11 %; catches noise less well than v2. See ml/CHECK30_V3.md.
CHECK_MIN_BEATS, CHECK_MIN_SQI, CHECK_IRR, CHECK_MIN_BSQI = 16, 0.4, 0.10, 0.8


def check(x: list[float], fs: int = 125, gate: bool = True) -> dict:
    """Verdict for a 30-s spot check: "regular", "irregular" (show a doctor) or "no_verdict" (repeat)."""
    a = analyze(x, fs)
    n, sqi, irr = len(a["r_peaks"]), a["quality"]["sqi"], a["irregularity"]
    b = bsqi(a["r_peaks"], r_peaks_amp(x, fs), fs)
    if n < CHECK_MIN_BEATS or sqi < CHECK_MIN_SQI or irr is None or (gate and b < CHECK_MIN_BSQI):
        verdict = "no_verdict"
    else:
        verdict = "irregular" if irr > CHECK_IRR else "regular"
    bpm = round(len(a["r_peaks"]) / (len(x) / fs) * 60) if x else None
    return {"verdict": verdict, "beats": n, "sqi": sqi, "irregularity": irr, "bsqi": b, "bpm": bpm,
            "reasons": a["quality"]["reasons"] + (["detectors_disagree"] if gate and b < CHECK_MIN_BSQI else [])}

In [ ]:
"""Early ventricular beats (PVC) from beat shape + timing — research estimate for the doctor, not shown to patients.

Pure Python, the same in the training notebook (ml/pvc_small.ipynb, pasted unchanged), on the server and, ported,
on the device (firmware/AFibMonitor/ectopy.h). A small tree ensemble trained on MIT-BIH DS1 and tested on DS2
(other people) decides "ventricular ectopic vs not" per beat. Counting PVCs is a medical-device claim: research only.
"""

import json
import math
import os
import statistics


HALF_S = 0.25            # beat window ±250 ms
N_PREV_SHAPES = 20       # median template of the previous beats
N_PREV_RR = 10           # local mean RR


def beat_features(x: list[float], peaks: list[int], fs: int = 125) -> list[tuple[int, list[float]]]:
    """(peak index, 39 features) for every beat that has 11 beats before it and one after it."""
    y = _highpass(x, fs)
    half = int(HALF_S * fs)
    n = len(y)
    shapes: list[list[float]] = []
    out = []
    for i in range(N_PREV_RR + 1, len(peaks) - 1):
        p = peaks[i]
        a, b = p - half, p + half + 1
        if a < 0 or b > n:
            continue
        w = y[a:b]
        med = statistics.median(w)
        prev_amp = [abs(y[q]) for q in peaks[max(0, i - N_PREV_SHAPES):i]]
        amp = statistics.median(prev_amp) if prev_amp else 0.0
        amp = amp if amp > 1e-6 else 1e-6
        seg = [(v - med) / amp for v in w]
        prev = shapes[-N_PREV_SHAPES:]
        tmpl = [statistics.median(col) for col in zip(*prev)] if len(prev) >= 5 else seg
        corr = _pearson(seg, tmpl)
        h = [abs(v) for v in seg[half - 12:half + 13]]
        hm = max(h)
        width = sum(1 for v in h if v >= 0.5 * hm) / fs if hm > 0 else 0.0
        pre = (peaks[i] - peaks[i - 1]) / fs
        post = (peaks[i + 1] - peaks[i]) / fs
        loc = sum((peaks[k] - peaks[k - 1]) / fs for k in range(i - N_PREV_RR, i)) / N_PREV_RR
        out.append((p, [pre, post, loc, pre / loc, post / pre, corr, width] + seg[::2]))
        shapes.append(seg)
    return out


def _pearson(a: list[float], b: list[float]) -> float:
    ma, mb = sum(a) / len(a), sum(b) / len(b)
    da = [u - ma for u in a]
    db = [v - mb for v in b]
    sa, sb = math.sqrt(sum(u * u for u in da)), math.sqrt(sum(v * v for v in db))
    if sa == 0 or sb == 0:
        return 0.0
    return sum(u * v for u, v in zip(da, db)) / (sa * sb)


# ---------------------------------------------------------------- model (exported from the notebook)
_MODEL_PATH = os.path.join(os.path.dirname(__file__), "pvc_model.json")
_MODEL = None


def load_model(path: str = _MODEL_PATH) -> dict | None:
    global _MODEL
    if _MODEL is None and os.path.exists(path):
        _MODEL = json.load(open(path))
    return _MODEL


def score(f: list[float], model: dict) -> float:
    """Probability that the beat is a ventricular ectopic beat.
    model = {"kind": "trees", "baseline": b, "trees": [[[feature, threshold, left, right, value, is_leaf], ...], ...]}
            {"kind": "logistic", "mean": [...], "scale": [...], "coef": [...], "intercept": c}"""
    if model["kind"] == "logistic":
        z = model["intercept"] + sum(c * (v - m) / s for c, v, m, s in zip(model["coef"], f, model["mean"], model["scale"]))
    else:
        z = model["baseline"]
        for nodes in model["trees"]:
            k = 0
            while not nodes[k][5]:
                feat, thr, left, right = nodes[k][0], nodes[k][1], nodes[k][2], nodes[k][3]
                k = left if f[int(feat)] <= thr else right
            z += nodes[k][4]
    return 1.0 / (1.0 + math.exp(-z))


def count_pvc(x: list[float], peaks: list[int], fs: int = 125, model: dict | None = None) -> dict | None:
    """{"beats": judged beats, "pvc": early ventricular beats, "pvc_at_s": [...]} or None without a model."""
    model = model or load_model()
    if not model:
        return None
    feats = beat_features(x, peaks, fs)
    hits = [p for p, f in feats if score(f, model) >= model.get("threshold", 0.5)]
    return {"beats": len(feats), "pvc": len(hits), "pvc_at_s": [round(p / fs, 2) for p in hits]}

In [ ]:
import gzip, base64
from multiprocessing import Pool
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
DS1 = [101,106,108,109,112,114,115,116,118,119,122,124,201,203,205,207,208,209,215,220,223,230]
DS2 = [100,103,105,111,113,117,121,123,200,202,210,212,213,214,219,221,222,228,231,232,233,234]
LAB = {**{s: 1 for s in 'VE'}, **{s: 0 for s in 'NLRejAaJS'}}
BEAT = set('NLRBAaJSVrFejnE/fQ?')
def to_125(sig, fs):
    x = sosfilt(butter(2, [0.5, 40], btype='band', fs=fs, output='sos'), np.nan_to_num(sig))
    from math import gcd
    g = gcd(int(fs), FS); return resample_poly(x, FS // g, int(fs) // g)

def rec_data(rec):
    r = wfdb.rdrecord(f'{DB}/{rec}', channels=[0]); ann = wfdb.rdann(f'{DB}/{rec}', 'atr')
    x = list(map(float, to_125(r.p_signal[:, 0], r.fs)))
    peaks = r_peaks(x, FS)
    ref_t = np.array([s * FS / r.fs for s, sym in zip(ann.sample, ann.symbol) if sym in BEAT])
    ref_y = np.array([LAB.get(sym, -1) for sym in ann.symbol if sym in BEAT])
    F, Y = [], []
    for p, f in beat_features(x, peaks, FS):
        j = int(np.clip(np.searchsorted(ref_t, p), 1, len(ref_t) - 1))
        k = j - 1 if abs(ref_t[j - 1] - p) < abs(ref_t[j] - p) else j
        y = ref_y[k] if abs(ref_t[k] - p) <= 0.15 * FS else -2
        if y >= 0: F.append(f); Y.append(y)
    return rec, np.array(F, np.float64), np.array(Y, int)

T0 = time.time()
with Pool(os.cpu_count()) as pool:
    DATA = {r: (f, y) for r, f, y in pool.map(rec_data, [r for r in DS1 + DS2 if os.path.exists(f'{DB}/{r}.dat')])}
print({r: (len(v[1]), int(v[1].sum())) for r, v in DATA.items()}, f'{time.time()-T0:.0f} s')

In [ ]:
def make(c):
    if c == 'A': return HistGradientBoostingClassifier(max_iter=50, max_depth=3, class_weight='balanced', random_state=42)
    if c == 'B': return HistGradientBoostingClassifier(max_iter=100, max_depth=4, class_weight='balanced', random_state=42)
    return 'logistic'
def fit(c, X, y):
    m = make(c)
    if m == 'logistic':
        sc = StandardScaler().fit(X); lr = LogisticRegression(class_weight='balanced', max_iter=2000).fit(sc.transform(X), y)
        return ('logistic', sc, lr)
    return ('trees', m.fit(X, y))
def proba(m, X):
    return m[2].predict_proba(m[1].transform(X))[:, 1] if m[0] == 'logistic' else m[1].predict_proba(X)[:, 1]
def f1(y, p):
    tp = int(((p == 1) & (y == 1)).sum()); fp = int(((p == 1) & (y == 0)).sum()); fn = int(((p == 0) & (y == 1)).sum())
    return 2 * tp / max(1, 2 * tp + fp + fn), tp / max(1, tp + fn), tp / max(1, tp + fp)

tr = [r for r in DS1 if r in DATA]; te = [r for r in DS2 if r in DATA]
X1 = np.concatenate([DATA[r][0] for r in tr]); y1 = np.concatenate([DATA[r][1] for r in tr])
g1 = np.concatenate([[r] * len(DATA[r][1]) for r in tr])
CV = {}
for c in 'ABC':
    P = np.zeros(len(y1), int)
    for a, b in GroupKFold(min(5, len(set(g1)))).split(X1, y1, g1):
        P[b] = (proba(fit(c, X1[a], y1[a]), X1[b]) >= 0.5).astype(int)
    CV[c] = f1(y1, P)
print('DS1 cross-validation (F1, sens, ppv):', {c: tuple(round(v, 4) for v in CV[c]) for c in CV})
CHOSEN = max('ABC', key=lambda c: CV[c][0])
print('chosen:', CHOSEN)

In [ ]:
X2 = np.concatenate([DATA[r][0] for r in te]); y2 = np.concatenate([DATA[r][1] for r in te])
RES = {}
for c in 'ABC':
    m = fit(c, X1, y1); p = (proba(m, X2) >= 0.5).astype(int)
    RES[c] = dict(zip(('F1', 'sens', 'ppv'), f1(y2, p)))
    if c == CHOSEN: MODEL, PRED = m, p
print('DS2:', {c: {k: round(v, 4) for k, v in RES[c].items()} for c in RES})

# export the chosen model to plain numbers and check it reproduces the probabilities
if MODEL[0] == 'logistic':
    EXP = {'kind': 'logistic', 'mean': MODEL[1].mean_.tolist(), 'scale': MODEL[1].scale_.tolist(),
           'coef': MODEL[2].coef_[0].tolist(), 'intercept': float(MODEL[2].intercept_[0]), 'threshold': 0.5}
else:
    clf = MODEL[1]; trees = []
    for it in clf._predictors:
        nd = it[0].nodes
        trees.append([[int(n['feature_idx']), float(n['num_threshold']), int(n['left']), int(n['right']), float(n['value']), int(n['is_leaf'])] for n in nd])
    EXP = {'kind': 'trees', 'baseline': float(np.ravel(clf._baseline_prediction)[0]), 'trees': trees, 'threshold': 0.5}
pe = np.array([score(list(f), EXP) for f in X2]); ps = proba(MODEL, X2)
print('export check: max |p_exported - p_sklearn| =', float(np.abs(pe - ps).max()), '| same decisions:', int(((pe >= .5) == (ps >= .5)).sum()), 'of', len(ps))

rows = []
i0 = 0
for r in te:
    n = len(DATA[r][1]); yy, pp = DATA[r][1], PRED[i0:i0 + n]; i0 += n
    rows.append(dict(record=r, beats=n, ref_V=int(yy.sum()), det_V=int(pp.sum()), burden_ref=yy.mean(), burden_det=pp.mean()))
R = pd.DataFrame(rows)
from scipy.stats import spearmanr
S = {'chosen': CHOSEN, 'cv_ds1': CV, 'ds2': RES, 'nodes': sum(len(t) for t in EXP.get('trees', [])) or len(EXP.get('coef', [])),
     'burden_spearman': spearmanr(R.burden_ref, R.burden_det).correlation,
     'records_V_ge10pct': int((R.burden_ref >= .1).sum()), 'found': int(((R.burden_ref >= .1) & (R.burden_det >= .1)).sum()),
     'false': int(((R.burden_ref < .1) & (R.burden_det >= .1)).sum())}
json.dump(S, open(f'{OUT_DIR}/pvc_small_summary.json', 'w'), indent=1, default=float)
json.dump(EXP, open(f'{OUT_DIR}/pvc_model.json', 'w'))
print(json.dumps(S, indent=1, default=lambda v: round(float(v), 4)))
print(R.assign(burden_ref=(R.burden_ref * 100).round(1), burden_det=(R.burden_det * 100).round(1)).to_string(index=False))
blob = base64.b64encode(gzip.compress(json.dumps(EXP).encode(), 9)).decode()
print('MODEL_SHA', __import__('hashlib').sha256(json.dumps(EXP).encode()).hexdigest()[:16], 'LEN', len(blob))
for i in range(0, len(blob), 3000): print('MODEL_B64', i // 3000, blob[i:i + 3000])